# Análise de sentimento **online** dos posts do X (@diariodonordeste) com **Spark Streaming**

**Objetivo:** coletar posts do **X (Twitter)** do **Diário do Nordeste** e, com **Apache Spark Structured Streaming**, classificar o **sentimento de cada notícia em tempo real** (POSITIVO / NEUTRO / NEGATIVO), mostrando o resultado à medida que os posts chegam.

| Etapa | Tecnologia |
|-------|------------|
| Fonte | posts do X (`@diariodonordeste`) via API v2 |
| Fonte alternativa | posts de exemplo (para rodar sem token) |
| Stream | pasta monitorada (1 post = 1 micro-batch) |
| Processamento | **Apache Spark Structured Streaming** |
| Sentimento | modelo multilíngue (HuggingFace) em PT-BR |
| Saída | sentimento por notícia + distribuição geral |

> ⚠️ **Runtime → Factory reset runtime** antes de executar, para garantir uma VM limpa.

### Sobre a coleta no X
A API do X (v2) exige um **Bearer Token**. Se você tiver um, cole na célula de configuração e o notebook busca os posts reais de `@diariodonordeste`. **Sem token**, o notebook usa um conjunto de **posts de exemplo** (manchetes no estilo do jornal) para o pipeline rodar de ponta a ponta no Colab.

## 1. Dependências + configuração da coleta

In [ ]:
%pip -q install pyspark==3.5.1 transformers torch requests pandas matplotlib

import os
import json
import time
import threading
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURAÇÃO DA COLETA NO X
# ============================================================
CONTA_X = "diariodonordeste"

# Cole aqui seu Bearer Token da API do X (opcional).
# No Colab pode usar: from google.colab import userdata; X_BEARER_TOKEN = userdata.get("X_BEARER_TOKEN")
X_BEARER_TOKEN = os.getenv("X_BEARER_TOKEN", "")

MAX_POSTS = 20

STREAM_DIR = Path("/content/posts_stream")
STREAM_DIR.mkdir(parents=True, exist_ok=True)

print("Conta X:", f"@{CONTA_X}")
print("Token configurado:", "SIM" if X_BEARER_TOKEN else "NÃO (usará posts de exemplo)")


## 2. Coletar os posts do X (com fallback para exemplos)

Se houver token, busca os posts reais via **API v2 do X**: primeiro o `id` da conta pelo username, depois os últimos tweets. Sem token (ou em caso de erro/limite), usa os posts de exemplo.

In [ ]:
# Posts de exemplo (manchetes no estilo Diário do Nordeste) — usados sem token
POSTS_EXEMPLO = [
    "Fortaleza inaugura novo hospital com 200 leitos e amplia atendimento no Ceará",
    "Chuvas fortes causam alagamentos e deixam famílias desabrigadas no interior",
    "Ceará bate recorde de energia solar e lidera geração renovável no Nordeste",
    "Acidente grave na BR-116 deixa feridos e interdita a rodovia por horas",
    "Turismo cresce no litoral cearense e aquece a economia local nas férias",
    "Assalto a banco assusta moradores no Centro de Fortaleza nesta manhã",
    "Estudantes do Ceará conquistam medalhas em olimpíada nacional de matemática",
    "Falta de água atinge bairros e população reclama do abastecimento irregular",
    "Novo parque é entregue à população e vira opção de lazer na capital",
    "Preço dos alimentos sobe e pesa no orçamento das famílias cearenses",
    "Seleção de vôlei do Ceará vence e garante vaga na final do campeonato",
    "Incêndio destrói galpão e mobiliza bombeiros na Região Metropolitana",
]


def coletar_posts_x():
    # Sem token: usa os exemplos
    if not X_BEARER_TOKEN:
        return [
            {"id": f"exemplo_{i}", "texto": t,
             "criado_em": datetime.utcnow().isoformat()}
            for i, t in enumerate(POSTS_EXEMPLO)
        ]

    headers = {"Authorization": f"Bearer {X_BEARER_TOKEN}"}
    try:
        # 1) username -> id
        u = requests.get(
            f"https://api.twitter.com/2/users/by/username/{CONTA_X}",
            headers=headers, timeout=20,
        )
        u.raise_for_status()
        user_id = u.json()["data"]["id"]

        # 2) últimos tweets do usuário
        r = requests.get(
            f"https://api.twitter.com/2/users/{user_id}/tweets",
            headers=headers,
            params={
                "max_results": MAX_POSTS,
                "tweet.fields": "created_at",
                "exclude": "retweets,replies",
            },
            timeout=20,
        )
        r.raise_for_status()
        tweets = r.json().get("data", [])
        return [
            {"id": t["id"], "texto": t["text"],
             "criado_em": t.get("created_at", datetime.utcnow().isoformat())}
            for t in tweets
        ]
    except Exception as e:
        print(f"⚠️  Falha ao coletar do X ({e}). Usando posts de exemplo.")
        return [
            {"id": f"exemplo_{i}", "texto": t,
             "criado_em": datetime.utcnow().isoformat()}
            for i, t in enumerate(POSTS_EXEMPLO)
        ]


posts = coletar_posts_x()
print(f"Coletados {len(posts)} posts.")
for p in posts[:3]:
    print("•", p["texto"][:90])


## 3. Pipeline de análise de sentimento (PT-BR)

Usamos o modelo multilíngue `nlptown/bert-base-multilingual-uncased-sentiment` (avaliação de 1 a 5 estrelas), que funciona bem em português. Mapeamos:

- **1–2 estrelas → NEGATIVO**
- **3 estrelas → NEUTRO**
- **4–5 estrelas → POSITIVO**

In [ ]:
from transformers import pipeline

sentimento = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
)


def classificar(texto: str):
    res = sentimento(texto[:512])[0]
    estrelas = int(res["label"][0])   # ex.: "4 stars" -> 4
    if estrelas <= 2:
        rotulo = "NEGATIVO"
    elif estrelas == 3:
        rotulo = "NEUTRO"
    else:
        rotulo = "POSITIVO"
    return rotulo, estrelas, round(float(res["score"]), 3)


# Teste rápido
for t in ["Ceará bate recorde e lidera energia solar", "Acidente grave deixa feridos na BR"]:
    print(classificar(t), "→", t)


## 4. Produtor — envia cada post para o stream

Cada post vira um arquivo JSON na pasta monitorada pelo Spark. Como usamos `maxFilesPerTrigger=1`, **cada notícia é processada em um micro-batch** — sentimento verdadeiramente *online*.

In [ ]:
INTERVALO_SEG = 1.5


def produzir_posts(lista, stop_event: threading.Event):
    for i, post in enumerate(lista):
        if stop_event.is_set():
            break
        arq = STREAM_DIR / f"post_{i:03d}.json"
        arq.write_text(json.dumps(post, ensure_ascii=False), encoding="utf-8")
        print(f"📰 enviado: {post['texto'][:70]}...")
        time.sleep(INTERVALO_SEG)
    print("🏁 Produtor encerrado.")


print(f"Serão enviados {len(posts)} posts, 1 a cada {INTERVALO_SEG}s.")


## 5. Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

spark = (
    SparkSession.builder
    .appName("SentimentoOnlineDiarioNordeste")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")


## 6. Structured Streaming — sentimento online de cada notícia

No `foreachBatch`, para **cada post** do micro-batch chamamos o classificador e registramos o sentimento em tempo real.

In [ ]:
resultados = []              # acumula o sentimento de cada notícia
contagem = {"POSITIVO": 0, "NEUTRO": 0, "NEGATIVO": 0}

EMOJI = {"POSITIVO": "🟢", "NEUTRO": "🟡", "NEGATIVO": "🔴"}


def analisar_batch(batch_df, batch_id: int):
    linhas = batch_df.collect()
    if not linhas:
        return
    for r in linhas:
        rotulo, estrelas, score = classificar(r["texto"])
        contagem[rotulo] += 1
        resultados.append({
            "id": r["id"],
            "texto": r["texto"],
            "sentimento": rotulo,
            "estrelas": estrelas,
            "confianca": score,
        })
        print(f"{EMOJI[rotulo]} [{rotulo}] ({score:.0%}) {r['texto'][:80]}")


schema = StructType([
    StructField("id", StringType(), True),
    StructField("texto", StringType(), False),
    StructField("criado_em", StringType(), True),
])

posts_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .json(str(STREAM_DIR))
)

query = (
    posts_df.writeStream
    .outputMode("append")
    .foreachBatch(analisar_batch)
    .option("checkpointLocation", "/content/checkpoint_sentimento")
    .trigger(processingTime="2 seconds")
    .start()
)

print(f"Streaming query id: {query.id}")


## 7. Rodar a coleta + análise ao vivo

In [ ]:
stop_event = threading.Event()
produtor = threading.Thread(target=produzir_posts, args=(posts, stop_event), daemon=True)
produtor.start()

try:
    tempo_total = int(len(posts) * INTERVALO_SEG) + 25
    print(f"⏳ Analisando por ~{tempo_total}s...")
    query.awaitTermination(timeout=tempo_total)
finally:
    stop_event.set()
    produtor.join(timeout=10)
    time.sleep(5)
    query.stop()
    print("✅ Análise em streaming finalizada.")


## 8. Sentimento de cada notícia

In [ ]:
if not resultados:
    print("Nenhuma notícia analisada. Rode as células anteriores novamente.")
else:
    df = pd.DataFrame(resultados)[["sentimento", "estrelas", "confianca", "texto"]]
    display(df)


## 9. Distribuição de sentimentos

In [ ]:
if resultados:
    cores = {"POSITIVO": "#2f855a", "NEUTRO": "#d69e2e", "NEGATIVO": "#c53030"}
    labels = list(contagem.keys())
    valores = [contagem[k] for k in labels]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.bar(labels, valores, color=[cores[k] for k in labels])
    ax1.set_title("Notícias por sentimento")
    ax1.set_ylabel("Qtd. de notícias")
    for i, v in enumerate(valores):
        ax1.text(i, v, str(v), ha="center", va="bottom", fontweight="bold")

    nao_zero = [(l, v) for l, v in zip(labels, valores) if v > 0]
    ax2.pie([v for _, v in nao_zero], labels=[l for l, _ in nao_zero],
            colors=[cores[l] for l, _ in nao_zero], autopct="%1.0f%%")
    ax2.set_title("Proporção de sentimentos")

    plt.tight_layout()
    plt.show()

    print("Resumo:", contagem)


## 10. (Opcional) Encerrar Spark

In [ ]:
try:
    query.stop()
except Exception:
    pass
spark.stop()
print("🛑 Spark encerrado.")


## 11. Deploy (fora do Colab)

Scripts prontos em [`deploy_sentimento/`](https://github.com/naubergois/SparkSteammingAulas/tree/main/deploy_sentimento):

```bash
cd deploy_sentimento
pip install -r requirements.txt

# Coleta posts do X (com token) ou usa exemplos, gravando em ./posts_stream
export X_BEARER_TOKEN="SEU_TOKEN"   # opcional
python producer.py

# Análise de sentimento online com Spark
spark-submit sentiment_stream_job.py
```

Sem `X_BEARER_TOKEN`, o produtor usa os posts de exemplo. Com token, coleta os posts reais de `@diariodonordeste` via API v2 do X.
